# penrose — a guided demo

**penrose is a falsification-first research referee for quantitative trading claims.** You give it a claim (from a paper, a generator, or yourself); it reconstructs and tests the claim under a power-aware robustness stack (deflated Sharpe, locked holdout, walk-forward, regime kill-lens) and returns a verdict — `kill`, `underpowered`, `watch`, or `research-supported`. It is designed to **not fool itself**.

This notebook runs **clean-room**: the core demo needs only `pip install .` (no API key, no external data). It reproduces the paper's strongest section — the *detector self-calibration* — live. One section (the Chen–Zimmermann literature experiment) is optional and downloads public academic data.

Honest framing throughout: penrose finds **no new alpha**. It quantifies how little survives proper testing — and, unusually, it **validates its own detector** before any verdict is trusted.


## 0. Setup
Point Python at the repo and import the (self-contained) verdict engine.


In [ ]:
import sys, os, subprocess, re
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'src')); sys.path.insert(0, ROOT)
import numpy as np, pandas as pd
from penrose.pipeline import p7_backtest as P7, stages
from penrose.brain import Claim
PY = sys.executable
def run(script, *args):
    r = subprocess.run([PY, os.path.join(ROOT,'scripts',script), *map(str,args)],
                       capture_output=True, text=True, cwd=ROOT)
    return '\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l and 'pct_change' not in l)
print('penrose loaded — clean-room (vendored stats; no key/external data for the core demo).')


## 1. Ground truth: the eval suite
Before trusting penrose on real data, we check it discriminates on **planted strategies with known-correct verdicts** — a robust edge must survive, an overfit/regime-fragile/noise strategy must die. This is the regression net (every fix in the repo adds a guard here).


In [ ]:
print(run('eval_suite.py').splitlines()[-1])   # e.g. '55/55 passed'


## 2. Calibrate the detector (the novel part)
A skeptic is only trustworthy if you know its error rates. We measure penrose's **specificity** (does it reject signals with no real edge?) on five recognized null environments — white noise, regime-switching vol, a **bid-ask-bounce microstructure trap**, a zero-alpha factor, and GARCH clustering — each generated fresh, then run through the *real* verdict engine.

A trustworthy falsifier must certify **none** of them.


In [ ]:
print(run('calibration_nulls.py', 30))


**Read it honestly:** 0/150 reach `research-supported`. The instructive nuance (in the paper): the *statistical* gates alone reject four of five nulls but leak on the bid-ask-bounce microstructure trap — which is eliminated by a realistic transaction cost. That's concrete evidence that measured costs are load-bearing, not cosmetic.


### 2b. Sensitivity — what edge can it actually detect?
Specificity is half the picture. We inject a **known** information coefficient (IC) into returns and sweep: where does penrose start certifying? Below its detection floor it honestly returns `underpowered` rather than a false `kill`.


In [ ]:
print(run('calibration_sensitivity.py', 8))


The floor is a **sample-power artifact**, not a fixed limit: it falls with more history and (cross-sectionally) with breadth, toward the realistic 0.02–0.05 IC range. penrose is deliberately conservative — most published 'edges' would not survive its bar, which is the point.


## 3. The brain's connections (advisory, never gating)
As penrose accumulates verdicts, it finds structure across them — shared failure modes, cross-domain links, and principles. **Hard rule: these *inform*, they never *gate*.** The corpus contextualizes a result for a human; it never auto-rejects a new idea (every new claim is still tested independently). Principles draw from **structural kills only** (never `underpowered`) and their confidence **decays with age** — a stale kill does not bar a new test.

Below we run the discovery engine on a small illustrative corpus.


In [ ]:
from penrose import brain_connect as BC
from penrose.brain_connect import Record
# illustrative verdicts across domains (in a real run these come from the verdict corpus)
demo = [
  Record('vol-c1','crypto-volatility','kill','in_sample_only','macro signal forecasts BTC vol',True,None,'2026-01-01'),
  Record('vol-c2','crypto-volatility','kill','in_sample_only','vol-of-vol predicts realized vol',True,None,'2026-01-01'),
  Record('vol-c3','crypto-volatility','kill','in_sample_only','funding predicts vol',True,None,'2026-01-01'),
  Record('fund-c1','funding-carry','kill','in_sample_only','delta-neutral funding carry',True,None,'2026-01-01'),
  Record('pm-c1','prediction-market','kill','regime_fragile','resolution-halt edge',True,None,'2026-01-01'),
  Record('eq-c1','crypto-equity','underpowered','below_detection_floor','crypto-equity corr',False,False,'2026-01-01'),
]
c = BC.discover(demo)
print('structural kills:', c.stats['n_structural_kills'], '| underpowered (excluded from priors):', c.stats['n_underpowered'])
print('\nPRINCIPLES (structural kills only, confidence decays):')
for p in c.principles: print(f"  {p['principle_id']}  n={p['n_observations']} conf={p['confidence']}")
print('\nCROSS-DOMAIN failure modes:')
for x in c.cross_domain: print(f"  '{x['kill_reason']}' spans {x['domains']}")
print('\n(advisory only — no verdict was created or changed)')


Notice the underpowered claim is **excluded** from principle-forming: *"we couldn't resolve it"* must never become *"this neighborhood is dead."* That guardrail is what keeps the corpus from preemptively discarding ideas that could work.


## 4. (Optional) Referee the published literature — Chen–Zimmermann
This reproduces the paper's literature experiment: run all **212 published Chen–Zimmermann anomalies** through penrose, deflating across the full search. **Optional** because it downloads ~20 MB of public academic data (the openassetpricing package, by the paper's authors). Uncomment to run.


In [ ]:
# --- optional: downloads public CZ data, runs the 212-anomaly referee (~1-2 min) ---
# subprocess.run([PY,'-m','pip','install','-q','openassetpricing'])
# import openassetpricing as oap; m=oap.OpenAP()
# df=m.dl_port('op','pandas'); ls=df[df['port']=='LS']
# panel=ls.assign(date=pd.to_datetime(ls['date'])).pivot_table(index='date',columns='signalname',values='ret')
# os.makedirs(os.path.expanduser('~/Development/penrose-data/literature/chen_zimmermann'),exist_ok=True)
# panel.to_parquet(os.path.expanduser('~/Development/penrose-data/literature/chen_zimmermann/ls_panel.parquet'))
# print(run('cz_referee.py'))
print('CZ experiment is optional — uncomment above to download + run. Result in the paper: ~48% per-anomaly,'
      ' down to ~3% deflated by the full 212-anomaly search.')


## What we did NOT reproduce here
The **RD-Agent generator experiment** (refereeing 16 GLM-generated factors: 14/16 killed per-factor, 0/16 deflated) needs a multi-hour external setup (Qlib + the RD-Agent agent loop), so it isn't notebook-reproducible. See the paper / `scripts/rdagent_referee.py` for the methodology and the handoff format.

## Summary
You ran, clean-room: the ground-truth eval, the detector self-calibration (5-null specificity + injection sensitivity), and the advisory connection-discovery — the paper's core, live. penrose's honest contribution is **integration + self-calibration**: an independent referee that validates its own detector and reports *dead vs underpowered vs broken*, not a single undifferentiated reject.

Paper: `docs/PENROSE_SYSTEMS_PAPER.md` · code: this repo · `make dash` for the live dashboard (incl. the Connections view).
